In [2]:
import pandas as pd

analytical_df = pd.read_csv(
    "./data/delivery_performance_cleaned.csv",
    parse_dates=["order_date", "expected_delivery_date", "actual_delivery_date"]
)
analytical_df.head()

,Unnamed: 0,order_id,warehouse_id,order_date,expected_delivery_date,actual_delivery_date,units
0,0,ORD_00064,WH_C,2026-03-10,2026-03-12,2026-03-12,14
1,1,ORD_02684,WH_C,2026-02-17,2026-02-20,2026-02-20,18
2,2,ORD_00103,WH_C,2026-03-08,2026-03-11,2026-03-12,15
3,3,ORD_02692,WH_D,2026-03-29,2026-04-02,2026-04-02,45
4,4,ORD_00417,WH_C,2026-03-07,2026-03-11,2026-03-11,17


### **Orders Segemnetion**

In [4]:
analytical_df["delivery_status"] = (
    analytical_df["actual_delivery_date"] <= analytical_df["expected_delivery_date"]
).map(
    {
        True: "On-Time",
        False: "Late"
    }
)
analytical_df.head()

,Unnamed: 0,order_id,warehouse_id,order_date,expected_delivery_date,actual_delivery_date,units,delivery_status
0,0,ORD_00064,WH_C,2026-03-10,2026-03-12,2026-03-12,14,On-Time
1,1,ORD_02684,WH_C,2026-02-17,2026-02-20,2026-02-20,18,On-Time
2,2,ORD_00103,WH_C,2026-03-08,2026-03-11,2026-03-12,15,Late
3,3,ORD_02692,WH_D,2026-03-29,2026-04-02,2026-04-02,45,On-Time
4,4,ORD_00417,WH_C,2026-03-07,2026-03-11,2026-03-11,17,On-Time


### **Overall On-Time Delivery (OTD) Rate**

In [5]:
ontime_orders = analytical_df[
    analytical_df["delivery_status"] == "On-Time"
].shape[0]

overall_otd = ontime_orders * 100 / analytical_df["order_id"].nunique()
overall_otd

71.64827816783685

In [7]:
otd_by_warehouse = (
    analytical_df
    .groupby("warehouse_id")
    .agg(
        total_orders=("order_id", "nunique"),
        ontime_orders=("delivery_status", lambda x: (x == "On-Time").sum())
    )
)

otd_by_warehouse["otd_rate"] = (
    otd_by_warehouse["ontime_orders"] / otd_by_warehouse["total_orders"]
)

otd_by_warehouse.to_csv("./output/otd_by_warehouse.csv")
otd_by_warehouse

,total_orders,ontime_orders,otd_rate
warehouse_id,,,
WH_A,261,242,0.927203
WH_B,359,310,0.863510
WH_C,582,444,0.762887
WH_D,734,514,0.700272
WH_E,1055,633,0.600000
